# LeetCode 3353 — Minimum Total Operations bằng Python

Link đề: https://leetcode.com/problems/minimum-total-operations/ · Độ khó: **Easy** · Chủ đề: Array (mảng), duyệt một lượt, cặp kề nhau

Skeleton LeetCode:

```python
class Solution:
    def minOperations(self, nums: List[int]) -> int:
        ...
```

**Ghi chú (bài Premium):** đây là bài **Premium** (🔒). Đề trong notebook được viết lại theo hiểu biết về bài (có đối chiếu với hai ví dụ và ràng buộc mà tài liệu tham khảo ghi), nên nếu có tài khoản Premium hãy **đối chiếu lại đề gốc** trên LeetCode. Mọi đáp án trong notebook đã được kiểm chứng bằng code, kể cả bằng một **bộ tìm kiếm BFS vét cạn** số thao tác tối thiểu trên các mảng nhỏ. Khi chạy ngoài LeetCode ta thêm `from typing import List`.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: một **thao tác** là chọn một **tiền tố** (prefix — `p` phần tử đầu mảng) và một số nguyên `k` (có thể âm) rồi **cộng `k` vào mọi phần tử của tiền tố**; tìm số thao tác **ít nhất** để **mọi phần tử bằng nhau**.
2. Nắm các công cụ Python: cắt lát `nums[:p]`, cặp kề `zip(nums, nums[1:])`, **đếm bằng `sum` của biểu thức boolean** (`True` cộng như `1`), `itertools.groupby`, danh sách hiệu kề (`gaps`), và `max`/`min`.
3. Đi từ cách **mô phỏng thật từng thao tác** → **đếm số cặp kề khác nhau** (vòng lặp) → **`zip` + `sum`** → **đếm số nhóm bằng `groupby`** → **mảng hiệu kề và đếm số phần tử khác `0`**.
4. Hiểu mấu chốt: thao tác trên tiền tố độ dài `p` **chỉ làm thay đổi đúng một hiệu kề** (`nums[p] - nums[p-1]`) nên mỗi thao tác sửa được **nhiều nhất một** cặp kề đang khác nhau; và ta luôn sửa được từng cặp bằng đúng một thao tác. Vậy đáp án = **số cặp kề `(nums[i-1], nums[i])` khác nhau**.
5. Tránh các lỗi kinh điển: trả `n - 1` (coi mọi cặp đều khác), đếm `len(set(nums)) - 1` (số giá trị khác nhau, sai với mảng như `[1,2,1]`), so sánh vòng tròn `nums[-1]` khi `i = 0`, và `IndexError` do `nums[i + 1]` ở phần tử cuối.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums`. Bạn có thể thực hiện **bất kỳ số thao tác nào**. Mỗi **thao tác**:

- Chọn một **tiền tố** của mảng (đoạn bắt đầu từ đầu mảng và kéo dài tới một vị trí nào đó);
- Chọn một số nguyên `k` (**có thể âm**) và **cộng `k` vào mỗi phần tử** trong tiền tố đã chọn.

Trả về **số thao tác tối thiểu** để **mọi phần tử** của mảng **bằng nhau**.

**Ví dụ 1**

- Input: `nums = [1,4,2]`
- Output: `2`
- Giải thích: thao tác 1: chọn tiền tố `[1, 4]` (độ dài 2), cộng `-2` → `[-1, 2, 2]`. Thao tác 2: chọn tiền tố `[-1]` (độ dài 1), cộng `3` → `[2, 2, 2]`. Cần tối thiểu `2` thao tác.

**Ví dụ 2**

- Input: `nums = [10,10,10]`
- Output: `0`
- Giải thích: mọi phần tử đã bằng nhau, không cần thao tác nào.

**Ràng buộc**

- `1 <= nums.length <= 10^5`
- `-10^9 <= nums[i] <= 10^9`

(Đây là bài Premium; xem ghi chú ở đầu notebook.)

In [1]:
from typing import List
from itertools import groupby
import random

examples = [
    ([1, 4, 2], 2),
    ([10, 10, 10], 0),
]

for nums, expected in examples:
    changes = sum(1 for i in range(1, len(nums)) if nums[i] != nums[i - 1])
    print(nums, "-> số cặp kề khác nhau =", changes, "(kỳ vọng", expected, ")")
    assert changes == expected

[1, 4, 2] -> số cặp kề khác nhau = 2 (kỳ vọng 2 )
[10, 10, 10] -> số cặp kề khác nhau = 0 (kỳ vọng 0 )


## 3. Kiến thức nền

### 3.1. Tiền tố (prefix) và cắt lát `nums[:p]`

**Tiền tố** độ dài `p` của mảng là `p` phần tử **đầu tiên**. Trong Python `nums[:p]` lấy các phần tử chỉ số `0..p-1`. Ví dụ `nums = [1,4,2]` có các tiền tố: `[1]`, `[1, 4]`, `[1, 4, 2]` (cả mảng cũng là một tiền tố).

In [2]:
nums = [1, 4, 2]
for p in range(1, len(nums) + 1):
    print(f"tiền tố độ dài {p}: {nums[:p]}")

tiền tố độ dài 1: [1]
tiền tố độ dài 2: [1, 4]
tiền tố độ dài 3: [1, 4, 2]


### 3.2. Thực hiện "cộng `k` vào tiền tố"

Để cài đặt thao tác ta duyệt `i` từ `0` đến `p - 1` và `arr[i] += k`. Nhớ: thay đổi trực tiếp **cập nhật tại chỗ** (in-place); nếu muốn giữ mảng gốc thì copy trước bằng `nums[:]` hoặc `list(nums)`.

In [3]:
def apply_prefix(arr, p, k):
    """Cộng k vào p phần tử đầu của arr (sửa tại chỗ)."""
    for i in range(p):
        arr[i] += k

arr = [1, 4, 2]
apply_prefix(arr, 2, -2)
print(arr)             # [-1, 2, 2]
apply_prefix(arr, 1, 3)
print(arr)             # [2, 2, 2]

[-1, 2, 2]
[2, 2, 2]


### 3.3. Cặp kề nhau: `zip(nums, nums[1:])`

`zip(a, b)` ghép từng phần tử của hai dãy theo vị trí, dừng khi dãy ngắn hết. `nums` và `nums[1:]` lệch nhau một vị trí, nên `zip(nums, nums[1:])` cho **mọi cặp kề** `(nums[0], nums[1])`, `(nums[1], nums[2])`, ...

In [4]:
nums = [1, 4, 2]
print(list(zip(nums, nums[1:])))        # [(1, 4), (4, 2)]
for x, y in zip(nums, nums[1:]):
    print(x, y, "khác nhau?", x != y)
print(list(zip([5], [5][1:])))          # []  -> mảng 1 phần tử không có cặp kề nào

[(1, 4), (4, 2)]
1 4 khác nhau? True
4 2 khác nhau? True
[]


### 3.4. Đếm bằng `sum` của biểu thức boolean

Trong Python `True` bằng `1` và `False` bằng `0` khi cộng. Vì vậy `sum(x != y for x, y in ...)` **đếm số cặp** thoả `x != y`. (Có thể viết `sum(1 for ... if ...)` cho dễ đọc hơn.)

In [5]:
print(True + True + False)               # 2
flags = [x != y for x, y in zip([1, 4, 2], [4, 2])]
print(flags, sum(flags))                 # [True, True] 2
print(sum(1 for x, y in zip([3, 3, 5], [3, 5]) if x != y))   # 1

2
[True, True] 2
1


### 3.5. Mảng hiệu kề `gaps[i] = nums[i+1] - nums[i]`

Mảng `gaps` (hiệu hai phần tử kề) cho biết mảng **phẳng** ở đâu: `gaps[i] == 0` nghĩa là `nums[i] == nums[i+1]`. **Mọi phần tử bằng nhau ⇔ mọi `gaps[i]` đều bằng `0`.** Đây là cách nhìn quan trọng để giải bài này.

In [6]:
def gaps_of(nums):
    return [nums[i + 1] - nums[i] for i in range(len(nums) - 1)]

print(gaps_of([1, 4, 2]))          # [3, -2]
print(gaps_of([10, 10, 10]))       # [0, 0]
print(gaps_of([2, 2, 2]))          # [0, 0]  -> mọi phần tử bằng nhau
print(gaps_of([5]))                # []

[3, -2]
[0, 0]
[0, 0]
[]


### 3.6. `itertools.groupby`: gom các phần tử liền nhau giống nhau

`groupby(nums)` gom các phần tử **liền kề bằng nhau** thành nhóm. Ví dụ `[3,3,5,5,5,2]` có `3` nhóm. Số chỗ đổi giá trị (số cặp kề khác nhau) luôn bằng **số nhóm trừ 1**.

In [7]:
nums = [3, 3, 5, 5, 5, 2]
groups = [(key, len(list(g))) for key, g in groupby(nums)]
print(groups)                              # [(3, 2), (5, 3), (2, 1)]
print(len(groups) - 1)                     # 2 chỗ đổi giá trị: 3->5 và 5->2

[(3, 2), (5, 3), (2, 1)]
2


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Xét `nums = [1, 4, 2]` với `gaps = [3, -2]` (hiệu kề). Muốn mọi phần tử bằng nhau, cần cả hai `gaps` bằng `0`.

**Một thao tác trên tiền tố độ dài `p` đổi gì?** Nó cộng cùng một `k` vào `nums[0..p-1]`. Mọi cặp kề **nằm cùng trong tiền tố** (cả hai đều tăng thêm `k`) giữ nguyên hiệu. Mọi cặp kề **nằm cùng ngoài tiền tố** cũng giữ nguyên. Chỉ cặp kề **bắc qua ranh giới** — `(nums[p-1], nums[p])` — đổi hiệu: `gaps[p-1]` giảm `k`. (Nếu `p = n`, cả mảng cùng tăng `k`, không `gaps` nào đổi.)

Vậy **mỗi thao tác đổi nhiều nhất một `gaps`**. Với `gaps = [3, -2]` có `2` số khác `0` nên cần **ít nhất 2 thao tác**; và ta làm được đúng `2` (sửa `gaps[1]` rồi `gaps[0]`, hoặc ngược lại). Kết luận: đáp án = số `gaps` khác `0`.

In [8]:
def show(arr, label):
    print(f"{label:<28} nums={arr}  gaps={gaps_of(arr)}")

arr = [1, 4, 2]
show(arr, "ban đầu")
apply_prefix(arr, 2, -2)       # p = 2: chỉ gaps[1] (cặp (4,2)) đổi
show(arr, "sau p=2, k=-2")
apply_prefix(arr, 1, 3)        # p = 1: chỉ gaps[0] đổi
show(arr, "sau p=1, k=+3")

arr = [1, 4, 2]
apply_prefix(arr, 3, 100)      # p = n: cộng cho cả mảng, gaps không đổi
show(arr, "p = n, k = 100")

ban đầu                      nums=[1, 4, 2]  gaps=[3, -2]
sau p=2, k=-2                nums=[-1, 2, 2]  gaps=[3, 0]
sau p=1, k=+3                nums=[2, 2, 2]  gaps=[0, 0]
p = n, k = 100               nums=[101, 104, 102]  gaps=[3, -2]


## 5. Cách 1 — Brute Force / baseline: mô phỏng thật từng thao tác

**Ý tưởng.** Làm đúng như đề nói, nhưng chọn thao tác theo chiến lược tham lam **từ phải sang trái**. Với `i` từ `n - 1` về `1`: nếu `nums[i - 1] != nums[i]`, ta thực hiện **một** thao tác: chọn tiền tố độ dài `i` (gồm `nums[0..i-1]`) và `k = nums[i] - nums[i - 1]`, để `nums[i - 1]` bằng `nums[i]`. Mỗi lần như vậy đếm `+1`.

Vì tiền tố `nums[0..i-1]` đều tăng `k` nên **các phần tử bên trái vẫn "liền mạch" với nhau**, không phá các cặp đã sửa (các cặp bên phải `i` đã bằng nhau từ trước và không bị đụng).

In [9]:
def min_ops_simulate(nums: List[int]) -> int:
    arr = list(nums)                      # giữ nguyên input
    ops = 0
    for i in range(len(arr) - 1, 0, -1):  # i = n-1, ..., 1
        if arr[i - 1] != arr[i]:
            k = arr[i] - arr[i - 1]
            for j in range(i):            # cộng k vào tiền tố độ dài i
                arr[j] += k
            ops += 1
    assert len(set(arr)) == 1             # kết quả: mọi phần tử bằng nhau
    return ops

print(min_ops_simulate([1, 4, 2]))        # 2
print(min_ops_simulate([10, 10, 10]))     # 0
print(min_ops_simulate([3, 3, 5, 5, 5, 2, 2, 7]))   # 3

2
0
3


**Độ phức tạp (Cách 1).** Mỗi thao tác cập nhật tối đa `n` phần tử và có tối đa `n - 1` thao tác: **Time `O(n²)`**, Space `O(n)` (bản sao mảng).

**Nút thắt.** Ta **không cần thật sự cộng** `k` vào tiền tố: phần quan trọng chỉ là **đếm** xem phải làm bao nhiêu thao tác. Lập luận ở mục 4 cho thấy đáp án chỉ phụ thuộc vào số cặp kề khác nhau — `O(n)` là đủ.

## 6. Ý tưởng thuật toán / bước đột phá

Hai nhận xét hợp lại:

1. **Cận dưới:** mỗi thao tác đổi **nhiều nhất một** `gaps[i]`, mà để thành "mọi phần tử bằng nhau" cần **mọi** `gaps[i] = 0`. Vậy số thao tác `>=` số `gaps[i] != 0`.
2. **Đạt được cận dưới:** với mỗi `gaps[i] != 0` ta dùng đúng một thao tác trên tiền tố độ dài `i + 1` (kéo `nums[0..i]` lên bằng `nums[i+1]`); thao tác này **chỉ** sửa `gaps[i]` về `0` và **không** làm hỏng các `gaps` khác. (Chính là mô phỏng của Cách 1.)

**Đáp án = số cặp `(nums[i], nums[i+1])` với `nums[i] != nums[i+1]`.** Một vòng lặp là đủ, `O(n)` thời gian, `O(1)` bộ nhớ. Ta không quan tâm giá trị cuối cùng hay độ lớn `k`.

In [10]:
nums = [3, 3, 5, 5, 5, 2, 2, 7]
print("gaps =", gaps_of(nums))
print("số gaps khác 0 =", sum(1 for g in gaps_of(nums) if g != 0))       # 3
print("đáp án mô phỏng =", min_ops_simulate(nums))                       # 3

gaps = [0, 2, 0, 0, -3, 0, 5]
số gaps khác 0 = 3
đáp án mô phỏng = 3


## 7. Các cách tiếp theo

### Cách 2 — Vòng lặp đếm số cặp kề khác nhau

**Giải thích từng đoạn.**

1. `ops = 0` — biến đếm số thao tác.
2. `for i in range(1, len(nums))` — mỗi `i` xét cặp `(nums[i-1], nums[i])`; bắt đầu từ `1` để **luôn có phần tử đứng trước** (tránh `nums[-1]`).
3. `if nums[i] != nums[i - 1]: ops += 1` — cặp khác nhau cần đúng một thao tác.
4. Trả `ops`.

In [11]:
def min_ops_loop(nums: List[int]) -> int:
    ops = 0
    for i in range(1, len(nums)):
        if nums[i] != nums[i - 1]:
            ops += 1
    return ops

print(min_ops_loop([1, 4, 2]))       # 2
print(min_ops_loop([10, 10, 10]))    # 0
print(min_ops_loop([7]))             # 0 (một phần tử)

2
0
0


In [12]:
# Code nộp LeetCode cho Cách 2 (đổi tên lớp thành Solution khi nộp)
class SolutionLoop:
    def minOperations(self, nums: List[int]) -> int:
        ops = 0
        for i in range(1, len(nums)):
            if nums[i] != nums[i - 1]:
                ops += 1
        return ops

print(SolutionLoop().minOperations([1, 4, 2]))   # 2

2


**Độ phức tạp Cách 2:** **Time `O(n)`**, **Space `O(1)`**.

### Cách 3 — `zip` + `sum` (một dòng)

**Giải thích.** `zip(nums, nums[1:])` cho mọi cặp kề; `sum(x != y for x, y in ...)` đếm số cặp khác nhau (mục 3.3–3.4). Lưu ý `nums[1:]` **tạo bản sao** nên tốn `O(n)` bộ nhớ phụ; trong Python 3.10+ có thể dùng `itertools.pairwise(nums)` để không sao chép.

In [13]:
from itertools import pairwise      # Python 3.10+

def min_ops_zip(nums: List[int]) -> int:
    return sum(x != y for x, y in zip(nums, nums[1:]))

def min_ops_pairwise(nums: List[int]) -> int:
    return sum(x != y for x, y in pairwise(nums))

print(min_ops_zip([1, 4, 2]), min_ops_pairwise([1, 4, 2]))            # 2 2
print(min_ops_zip([10, 10, 10]), min_ops_pairwise([10, 10, 10]))      # 0 0

2 2
0 0


In [14]:
# Code nộp LeetCode cho Cách 3
from itertools import pairwise

class SolutionZip:
    def minOperations(self, nums: List[int]) -> int:
        return sum(x != y for x, y in pairwise(nums))

print(SolutionZip().minOperations([3, 3, 5, 5, 5, 2, 2, 7]))   # 3

3


**Độ phức tạp Cách 3:** `zip(nums, nums[1:])`: **Time `O(n)`, Space `O(n)`** (bản sao `nums[1:]`); `pairwise`: **Time `O(n)`, Space `O(1)`**.

### Cách 4 — Đếm số nhóm liền nhau bằng `groupby`

**Giải thích.** Mỗi lần đổi giá trị là một "ranh giới nhóm". Nếu mảng có `g` nhóm giá trị liền nhau thì có `g - 1` chỗ đổi giá trị, tức đáp án `g - 1`. Cách này hợp khi bạn nghĩ theo "đoạn bằng nhau".

In [15]:
def min_ops_groupby(nums: List[int]) -> int:
    groups = sum(1 for _ in groupby(nums))     # số nhóm liền nhau bằng nhau
    return groups - 1

print(min_ops_groupby([1, 4, 2]))                   # 3 nhóm -> 2
print(min_ops_groupby([10, 10, 10]))                # 1 nhóm -> 0
print(min_ops_groupby([3, 3, 5, 5, 5, 2, 2, 7]))    # 4 nhóm -> 3

2
0
3


In [16]:
# Code nộp LeetCode cho Cách 4
from itertools import groupby

class SolutionGroupby:
    def minOperations(self, nums: List[int]) -> int:
        return sum(1 for _ in groupby(nums)) - 1

print(SolutionGroupby().minOperations([1, 4, 2]))   # 2

2


**Độ phức tạp Cách 4:** **Time `O(n)`**, **Space `O(1)`** (`groupby` là iterator lười).

### Cách 5 — Mảng hiệu kề `gaps` rồi đếm phần tử khác `0`

**Giải thích.** Dựng `gaps[i] = nums[i+1] - nums[i]` (mục 3.5) rồi đếm các phần tử khác `0`. Cách này bám sát **chứng minh** (mỗi thao tác đổi đúng một `gaps`) nên rất hợp để hiểu bản chất, dù tốn thêm `O(n)` bộ nhớ.

In [17]:
def min_ops_gaps(nums: List[int]) -> int:
    gaps = [nums[i + 1] - nums[i] for i in range(len(nums) - 1)]
    return sum(1 for g in gaps if g != 0)

print(min_ops_gaps([1, 4, 2]))                   # 2
print(min_ops_gaps([10, 10, 10]))                # 0
print(min_ops_gaps([-10**9, 10**9, -10**9]))     # 2 (giá trị biên)

2
0
2


In [18]:
# Code nộp LeetCode cho Cách 5
class SolutionGaps:
    def minOperations(self, nums: List[int]) -> int:
        gaps = [nums[i + 1] - nums[i] for i in range(len(nums) - 1)]
        return sum(1 for g in gaps if g != 0)

print(SolutionGaps().minOperations([1, 4, 2]))   # 2

2


**Độ phức tạp Cách 5:** **Time `O(n)`**, **Space `O(n)`** (mảng `gaps`).

## 8. TRACE từng bước

Trace hai thứ: (a) **mô phỏng thao tác thật** (Cách 1) trên `[1, 4, 2]` — để thấy từng thao tác và mảng sau mỗi bước; (b) **vòng lặp đếm** (Cách 2) trên `[3, 3, 5, 5, 5, 2, 2, 7]`.

In [19]:
def trace_simulate(nums: List[int]) -> int:
    arr = list(nums)
    ops = 0
    print("mảng ban đầu:", arr)
    for i in range(len(arr) - 1, 0, -1):
        print(f"i={i}: so arr[{i-1}]={arr[i-1]} với arr[{i}]={arr[i]}", end="")
        if arr[i - 1] != arr[i]:
            k = arr[i] - arr[i - 1]
            for j in range(i):
                arr[j] += k
            ops += 1
            print(f" -> khác nhau: chọn tiền tố độ dài {i}, k={k:+d}, mảng = {arr}, ops={ops}")
        else:
            print(f" -> bằng nhau, bỏ qua, ops={ops}")
    print("kết quả:", arr, "số thao tác =", ops)
    return ops

trace_simulate([1, 4, 2])
print()
trace_simulate([3, 3, 5, 5, 5, 2, 2, 7])

mảng ban đầu: [1, 4, 2]
i=2: so arr[1]=4 với arr[2]=2 -> khác nhau: chọn tiền tố độ dài 2, k=-2, mảng = [-1, 2, 2], ops=1
i=1: so arr[0]=-1 với arr[1]=2 -> khác nhau: chọn tiền tố độ dài 1, k=+3, mảng = [2, 2, 2], ops=2
kết quả: [2, 2, 2] số thao tác = 2

mảng ban đầu: [3, 3, 5, 5, 5, 2, 2, 7]
i=7: so arr[6]=2 với arr[7]=7 -> khác nhau: chọn tiền tố độ dài 7, k=+5, mảng = [8, 8, 10, 10, 10, 7, 7, 7], ops=1
i=6: so arr[5]=7 với arr[6]=7 -> bằng nhau, bỏ qua, ops=1
i=5: so arr[4]=10 với arr[5]=7 -> khác nhau: chọn tiền tố độ dài 5, k=-3, mảng = [5, 5, 7, 7, 7, 7, 7, 7], ops=2
i=4: so arr[3]=7 với arr[4]=7 -> bằng nhau, bỏ qua, ops=2
i=3: so arr[2]=7 với arr[3]=7 -> bằng nhau, bỏ qua, ops=2
i=2: so arr[1]=5 với arr[2]=7 -> khác nhau: chọn tiền tố độ dài 2, k=+2, mảng = [7, 7, 7, 7, 7, 7, 7, 7], ops=3
i=1: so arr[0]=7 với arr[1]=7 -> bằng nhau, bỏ qua, ops=3
kết quả: [7, 7, 7, 7, 7, 7, 7, 7] số thao tác = 3


3

In [20]:
def trace_loop(nums: List[int]) -> int:
    ops = 0
    print("nums =", nums)
    for i in range(1, len(nums)):
        diff = nums[i] != nums[i - 1]
        if diff:
            ops += 1
        print(f"i={i}: nums[{i-1}]={nums[i-1]}, nums[{i}]={nums[i]}, khác? {diff}, ops={ops}")
    return ops

trace_loop([3, 3, 5, 5, 5, 2, 2, 7])

nums = [3, 3, 5, 5, 5, 2, 2, 7]
i=1: nums[0]=3, nums[1]=3, khác? False, ops=0
i=2: nums[1]=3, nums[2]=5, khác? True, ops=1
i=3: nums[2]=5, nums[3]=5, khác? False, ops=1
i=4: nums[3]=5, nums[4]=5, khác? False, ops=1
i=5: nums[4]=5, nums[5]=2, khác? True, ops=2
i=6: nums[5]=2, nums[6]=2, khác? False, ops=2
i=7: nums[6]=2, nums[7]=7, khác? True, ops=3


3

Bảng trace Cách 2 với `nums = [3, 3, 5, 5, 5, 2, 2, 7]`:

| i | `nums[i-1]` | `nums[i]` | khác? | `ops` sau |
|---|---|---|---|---|
| 1 | 3 | 3 | không | 0 |
| 2 | 3 | 5 | **có** | 1 |
| 3 | 5 | 5 | không | 1 |
| 4 | 5 | 5 | không | 1 |
| 5 | 5 | 2 | **có** | 2 |
| 6 | 2 | 2 | không | 2 |
| 7 | 2 | 7 | **có** | 3 |

Đáp án `3` — trùng với số nhóm `[3,3] [5,5,5] [2,2] [7]` trừ `1`.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 2 (vòng lặp): rõ nhất và chạy `O(n)`, `O(1)` bộ nhớ.

In [21]:
class Solution:
    def minOperations(self, nums: List[int]) -> int:
        ops = 0
        for i in range(1, len(nums)):
            if nums[i] != nums[i - 1]:
                ops += 1
        return ops

sol = Solution()
print(sol.minOperations([1, 4, 2]))        # 2
print(sol.minOperations([10, 10, 10]))     # 0

2
0


## 10. Vì sao thuật toán đúng

Gọi `D` là số chỉ số `i` (`0 <= i < n - 1`) có `nums[i] != nums[i+1]`.

- **Cận dưới `ops >= D`:** thao tác trên tiền tố độ dài `p` cộng cùng `k` vào `nums[0..p-1]`. Với mọi `i` mà `i, i+1` cùng `< p` hay cùng `>= p`, hiệu `nums[i+1] - nums[i]` **không đổi**. Chỉ `i = p - 1` (khi `p < n`) bị đổi. Vậy mỗi thao tác **thay đổi nhiều nhất một hiệu kề**. Đích đến: mọi hiệu kề bằng `0`. Có `D` hiệu khác `0` nên cần ít nhất `D` thao tác.
- **Đạt được `ops = D`:** duyệt `i` từ phải sang trái (như Cách 1). Với `i` mà `arr[i-1] != arr[i]`: dùng tiền tố độ dài `i` và `k = arr[i] - arr[i-1]`; sau đó `arr[i-1] == arr[i]`. Thao tác này chỉ đổi hiệu giữa vị trí `i-1` và `i` (đưa về `0`), các hiệu khác giữ nguyên, nên số hiệu khác `0` giảm đúng `1`. Lặp `D` lần là xong.
- **Không phụ thuộc giá trị:** đáp án chỉ cần biết hai phần tử kề có bằng nhau không, nên kể cả giá trị `±10^9` cũng không ảnh hưởng (Python không tràn số).
- **Mảng 1 phần tử:** không có cặp kề, `D = 0`, đã "bằng nhau" từ đầu.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1 — mô phỏng thao tác | `O(n²)` | `O(n)` | chứng minh bằng hành động |
| 2 — vòng lặp đếm | `O(n)` | `O(1)` | chuẩn, dễ đọc |
| 3 — `zip` + `sum` | `O(n)` | `O(n)` (`pairwise`: `O(1)`) | một dòng |
| 4 — `groupby` | `O(n)` | `O(1)` | nghĩ theo "số nhóm − 1" |
| 5 — mảng `gaps` | `O(n)` | `O(n)` | bám chứng minh |

Với `n` tới `10^5`, **Cách 1 (`O(n²)` ≈ `10^10`) sẽ quá chậm**, các cách còn lại chạy gần như tức thì.

## 12. Bộ test

Ta gom các cách vào `impls`. Ngoài ra viết một **oracle độc lập** bằng **BFS**: với mảng nhỏ, tìm số thao tác tối thiểu bằng cách thử **mọi** thao tác `(p, k)` trong một miền giá trị nhỏ. Oracle này không dùng bất kỳ suy luận nào ở trên, nên là một kiểm chứng tốt cho kết luận "đếm cặp kề khác nhau".

In [22]:
from collections import deque

def min_ops_bfs(nums: List[int], lo: int, hi: int) -> int:
    """Số thao tác tối thiểu, tìm kiếm BFS trong miền giá trị [lo, hi] và k trong [-(hi-lo), hi-lo]."""
    start = tuple(nums)
    if len(set(start)) == 1:
        return 0
    span = hi - lo
    seen = {start}
    dq = deque([(start, 0)])
    while dq:
        state, d = dq.popleft()
        for p in range(1, len(state) + 1):
            for k in range(-span, span + 1):
                if k == 0:
                    continue
                new = tuple(v + k if i < p else v for i, v in enumerate(state))
                if min(new) < lo or max(new) > hi:
                    continue
                if new in seen:
                    continue
                if len(set(new)) == 1:
                    return d + 1
                seen.add(new)
                dq.append((new, d + 1))
    raise AssertionError("không tìm được trong miền giới hạn")

print(min_ops_bfs([1, 4, 2], 0, 5))        # 2
print(min_ops_bfs([10, 10, 10], 9, 11))    # 0
print(min_ops_bfs([0, 1, 0, 1], 0, 2))     # 3

impls = {
    "simulate": min_ops_simulate,
    "loop": min_ops_loop,
    "zip": min_ops_zip,
    "pairwise": min_ops_pairwise,
    "groupby": min_ops_groupby,
    "gaps": min_ops_gaps,
    "Solution": Solution().minOperations,
    "SolutionLoop": SolutionLoop().minOperations,
    "SolutionZip": SolutionZip().minOperations,
    "SolutionGroupby": SolutionGroupby().minOperations,
    "SolutionGaps": SolutionGaps().minOperations,
}

def check_all(nums, expected, note=""):
    for name, f in impls.items():
        got = f(list(nums))
        assert got == expected, (name, nums, got, expected)
    print(f"OK  {nums if len(nums) <= 10 else str(nums[:10]) + '...'} -> {expected}  {note}")

check_all([1, 4, 2], 2, "ví dụ 1")
check_all([10, 10, 10], 0, "ví dụ 2")

2
0
3
OK  [1, 4, 2] -> 2  ví dụ 1
OK  [10, 10, 10] -> 0  ví dụ 2


### 12.1. Các trường hợp biên

In [23]:
check_all([5], 0, "một phần tử")
check_all([1, 2], 1, "hai phần tử khác nhau")
check_all([7, 7], 0, "hai phần tử bằng nhau")
check_all([1, 2, 1], 2, "đối xứng: có 2 giá trị khác nhau nhưng cần 2 thao tác (không phải 1)")
check_all([1, 2, 3, 4, 5], 4, "mọi cặp kề khác nhau: n - 1")
check_all([2, 2, 2, 2, 3], 1, "chỉ phần tử cuối khác")
check_all([3, 2, 2, 2, 2], 1, "chỉ phần tử đầu khác")
check_all([-10**9, 10**9], 1, "giá trị biên")
check_all([10**9, 10**9, 10**9], 0, "biên: toàn 10^9")
check_all([0, 0, 1, 1, 0, 0], 2, "hai chỗ đổi giá trị (tự dựng)")

OK  [5] -> 0  một phần tử
OK  [1, 2] -> 1  hai phần tử khác nhau
OK  [7, 7] -> 0  hai phần tử bằng nhau
OK  [1, 2, 1] -> 2  đối xứng: có 2 giá trị khác nhau nhưng cần 2 thao tác (không phải 1)
OK  [1, 2, 3, 4, 5] -> 4  mọi cặp kề khác nhau: n - 1
OK  [2, 2, 2, 2, 3] -> 1  chỉ phần tử cuối khác
OK  [3, 2, 2, 2, 2] -> 1  chỉ phần tử đầu khác
OK  [-1000000000, 1000000000] -> 1  giá trị biên
OK  [1000000000, 1000000000, 1000000000] -> 0  biên: toàn 10^9
OK  [0, 0, 1, 1, 0, 0] -> 2  hai chỗ đổi giá trị (tự dựng)


In [24]:
# n lớn: bỏ riêng Cách 1 (O(n^2)) khỏi phép so sánh
big = list(range(10**5))
for name, f in impls.items():
    if name == "simulate":
        continue
    assert f(big) == 10**5 - 1, name
alt = [i % 2 for i in range(10**5)]
for name, f in impls.items():
    if name == "simulate":
        continue
    assert f(alt) == 10**5 - 1, name
const = [3] * 10**5
for name, f in impls.items():
    if name == "simulate":
        continue
    assert f(const) == 0, name
print("n = 10^5: OK (loại Cách 1 vì O(n^2))")

n = 10^5: OK (loại Cách 1 vì O(n^2))


### 12.2. Phản ví dụ cho các cách làm sai

Ba cách "gần đúng": (a) trả `n - 1` luôn, (b) `len(set(nums)) - 1` (số giá trị khác nhau trừ 1), (c) so sánh vòng tròn (`nums[i] != nums[i-1]` với `i` từ `0`, dùng `nums[-1]`).

In [25]:
def wrong_n_minus_1(nums):
    return len(nums) - 1

def wrong_distinct(nums):
    return len(set(nums)) - 1

def wrong_circular(nums):
    return sum(1 for i in range(len(nums)) if nums[i] != nums[i - 1])   # i = 0 so với nums[-1]

for w in (wrong_n_minus_1, wrong_distinct, wrong_circular):
    for nums in ([1, 1], [1, 2, 1], [1, 2]):
        print(f"{w.__name__:<18} {nums} -> {w(nums)}  (đúng: {Solution().minOperations(nums)})")
    print()

assert wrong_n_minus_1([1, 1]) != Solution().minOperations([1, 1])
assert wrong_distinct([1, 2, 1]) != Solution().minOperations([1, 2, 1])
assert wrong_circular([1, 2]) != Solution().minOperations([1, 2])

wrong_n_minus_1    [1, 1] -> 1  (đúng: 0)
wrong_n_minus_1    [1, 2, 1] -> 2  (đúng: 2)
wrong_n_minus_1    [1, 2] -> 1  (đúng: 1)

wrong_distinct     [1, 1] -> 0  (đúng: 0)
wrong_distinct     [1, 2, 1] -> 1  (đúng: 2)
wrong_distinct     [1, 2] -> 1  (đúng: 1)

wrong_circular     [1, 1] -> 0  (đúng: 0)
wrong_circular     [1, 2, 1] -> 2  (đúng: 2)
wrong_circular     [1, 2] -> 2  (đúng: 1)



### 12.3. Đối chiếu với BFS vét cạn và ngẫu nhiên (có seed)

**(1)** Mọi mảng độ dài `1..5` với giá trị `0..2` (miền BFS `lo = 0, hi = 2`): so các cách với oracle BFS.
**(2)** Ngẫu nhiên có seed: độ dài `1..200`, giá trị trong `[-10^9, 10^9]` và cả giá trị nhỏ để có nhiều phần tử trùng nhau; so mọi cách với nhau (và `simulate`).

In [26]:
import itertools

total = 0
for n in range(1, 6):
    for tup in itertools.product(range(3), repeat=n):
        nums = list(tup)
        expected = min_ops_bfs(nums, 0, 2)
        for name, f in impls.items():
            assert f(nums) == expected, (name, nums, expected)
        total += 1
print("BFS vét cạn:", total, "mảng nhỏ khớp mọi cách: OK")

rng = random.Random(3353)
for trial in range(1500):
    n = rng.randint(1, 200)
    if rng.random() < 0.5:
        nums = [rng.randint(-10**9, 10**9) for _ in range(n)]
    else:
        nums = [rng.randint(-2, 2) for _ in range(n)]
    results = {name: f(nums) for name, f in impls.items()}
    assert len(set(results.values())) == 1, (nums, results)
print("ngẫu nhiên có seed: OK")

BFS vét cạn: 363 mảng nhỏ khớp mọi cách: OK


ngẫu nhiên có seed: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Trả `n - 1` (coi mọi cặp đều cần sửa)

In [27]:
def sai_n_tru_1(nums):
    return len(nums) - 1        # SAI

nums = [10, 10, 10]
print(sai_n_tru_1(nums), "(sai) vs", Solution().minOperations(nums), "(đúng)")
assert sai_n_tru_1(nums) == 2 and Solution().minOperations(nums) == 0

2 (sai) vs 0 (đúng)


**Vì sao sai:** nếu hai phần tử kề đã bằng nhau thì **không cần thao tác** cho cặp đó. Đếm phải có điều kiện `nums[i] != nums[i-1]`.

### Lỗi 2: Đếm số giá trị khác nhau `len(set(nums)) - 1`

In [28]:
def sai_set(nums):
    return len(set(nums)) - 1   # SAI

nums = [1, 2, 1]
print(sai_set(nums), "(sai) vs", Solution().minOperations(nums), "(đúng)")
assert sai_set(nums) == 1 and Solution().minOperations(nums) == 2

1 (sai) vs 2 (đúng)


**Vì sao sai:** thao tác tác động lên **tiền tố liên tiếp**, nên **vị trí** quan trọng, không chỉ tập giá trị. `[1, 2, 1]`: có hai ranh giới (`1|2` và `2|1`), mỗi ranh giới cần một thao tác dù chỉ có `2` giá trị khác nhau.

### Lỗi 3: So sánh vòng tròn với `nums[-1]`

In [29]:
def sai_vong_tron(nums):
    ops = 0
    for i in range(len(nums)):          # SAI: i = 0 so với nums[-1] (phần tử cuối!)
        if nums[i] != nums[i - 1]:
            ops += 1
    return ops

nums = [1, 2]
print(sai_vong_tron(nums), "(sai) vs", Solution().minOperations(nums), "(đúng)")
assert sai_vong_tron(nums) == 2 and Solution().minOperations(nums) == 1

2 (sai) vs 1 (đúng)


**Vì sao sai:** trong Python `nums[-1]` là **phần tử cuối**. Với `i = 0`, `nums[i - 1] = nums[-1]` so đầu với cuối — một "cặp" không tồn tại trong bài. **Sửa:** bắt đầu từ `i = 1`.

### Lỗi 4: `IndexError` khi dùng `nums[i + 1]`

In [30]:
def sai_index_error(nums):
    ops = 0
    for i in range(len(nums)):          # SAI: i = n-1 thì nums[i + 1] vượt biên
        if nums[i] != nums[i + 1]:
            ops += 1
    return ops

try:
    sai_index_error([1, 2, 3])
except IndexError as e:
    print("IndexError:", e)

def dung_range_tru_1(nums):
    return sum(1 for i in range(len(nums) - 1) if nums[i] != nums[i + 1])

print(dung_range_tru_1([1, 2, 3]))      # 2
assert dung_range_tru_1([1, 2, 3]) == 2

IndexError: list index out of range
2


**Vì sao sai:** ở `i = len(nums) - 1`, `nums[i + 1]` không tồn tại. **Sửa:** `range(len(nums) - 1)` nếu so với `nums[i + 1]`, hoặc `range(1, len(nums))` nếu so với `nums[i - 1]`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **Thao tác trên tiền tố/hậu tố/đoạn → hãy nghĩ tới mảng hiệu kề** (difference array). Thao tác "cộng `k` vào đoạn" chỉ đổi **hai** hiệu ở hai đầu đoạn (tiền tố: chỉ **một** hiệu).
- **"Làm cho mọi phần tử bằng nhau" ⇔ mọi hiệu kề bằng `0`.** Bài "bằng nhau" thường quy về đếm số hiệu khác `0`.
- **Đếm số lần đổi giá trị = số nhóm − 1.** Dùng cho: "Minimum Number of Flips ...", "Number of Segments", "Count Binary Substrings".
- **Kiểm tra cặp `(nums[i-1], nums[i])`** → luôn nhớ biên `i = 0` hay `i = n - 1`; đừng dùng `nums[-1]` vô tình.
- **Bài Premium không xem được đề?** Nếu thấy ví dụ nhỏ, hãy **tự rút quy luật bằng BFS vét cạn** trên mảng nhỏ (như mục 12) rồi mới tin vào công thức.

## 15. Biến thể của bài

1. **Thao tác trên hậu tố** (suffix) thay vì tiền tố: đáp án vẫn là số cặp kề khác nhau (đối xứng).
2. **Thao tác trên một đoạn bất kỳ** `[l, r]`: mỗi thao tác đổi tối đa **hai** hiệu kề nên cận dưới là `ceil(D / 2)`, nhưng có thể cần nhiều hơn — bài khó hơn, đừng áp dụng nguyên công thức của bài này.
3. **`k` chỉ được `+1`** (hoặc chỉ dương) — cần tổng giá trị các hiệu, không chỉ đếm.
4. **Làm mảng không giảm** (chỉ cần `nums[i] <= nums[i+1]`): chỉ đếm các hiệu âm.

In [31]:
def min_ops_suffix(nums: List[int]) -> int:
    """Biến thể: thao tác trên HẬU TỐ -> vẫn là số cặp kề khác nhau (đối xứng)."""
    return sum(1 for i in range(len(nums) - 1) if nums[i] != nums[i + 1])

print(min_ops_suffix([1, 4, 2]))     # 2

def min_ops_unit_steps(nums: List[int]) -> int:
    """Biến thể: mỗi thao tác tiền tố chỉ được cộng +1 hoặc -1: tổng |hiệu kề|."""
    return sum(abs(nums[i + 1] - nums[i]) for i in range(len(nums) - 1))

print(min_ops_unit_steps([1, 4, 2]))   # |4-1| + |2-4| = 3 + 2 = 5
print(min_ops_unit_steps([10, 10, 10]))   # 0

2
5
0


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1 | Mô phỏng thật từng thao tác (phải → trái) | `O(n²)` | `O(n)` | để **thấy** vì sao đếm đủ; chỉ cho `n` nhỏ |
| 2 | Vòng lặp đếm cặp kề khác nhau | `O(n)` | `O(1)` | chuẩn nộp bài |
| 3 | `zip`/`pairwise` + `sum` | `O(n)` | `O(n)` / `O(1)` | viết một dòng |
| 4 | `groupby`: số nhóm − 1 | `O(n)` | `O(1)` | nghĩ theo đoạn bằng nhau |
| 5 | Mảng `gaps`, đếm khác `0` | `O(n)` | `O(n)` | giảng giải/chứng minh |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `[1, 4, 2][:2]`, `list(zip([1, 2, 3], [1, 2, 3][1:]))`, `sum([True, False, True])`, `Solution().minOperations([5, 5, 5, 5])`, `Solution().minOperations([2, 9, 9, 2])`.

**Bài 2 (tính tay).** Với `nums = [4, 4, 1, 7, 7, 7, 3]`: liệt kê `gaps`, các cặp kề khác nhau và số thao tác tối thiểu; rồi viết một dãy thao tác `(p, k)` cụ thể làm mảng bằng nhau.

**Bài 3 (điền TODO).** Hoàn thành `min_operations_todo(nums)` theo Cách 2.

In [32]:
def min_operations_todo(nums):
    # TODO 1: ops = 0
    # TODO 2: duyệt i từ 1 tới len(nums) - 1
    # TODO 3: nếu nums[i] != nums[i - 1] thì ops += 1
    # TODO 4: trả về ops
    pass

# Tự làm xong rồi mới mở test:
# assert min_operations_todo([1, 4, 2]) == 2
# assert min_operations_todo([10, 10, 10]) == 0

**Bài 4 (tự viết lại).** Viết `min_operations_groupby2(nums)` dùng `itertools.groupby` nhưng **đếm số chỗ đổi giá trị bằng cách khác**: dùng `len(list(...))` rồi trừ `1`; sau đó viết thêm `ops_sequence(nums)` **trả về danh sách thao tác** `[(p, k), ...]` (theo chiến lược phải → trái của Cách 1) và kiểm tra rằng áp dụng chúng vào `nums` làm mọi phần tử bằng nhau.

**Bài 5 (biến thể / mini challenge).** Cho phép thao tác trên **hậu tố** (`p` phần tử **cuối**). Viết `min_ops_suffix_sim(nums)` mô phỏng (trái → phải) và chứng minh bằng test ngẫu nhiên rằng nó luôn bằng `Solution().minOperations(nums)`. Thử thêm: nếu thao tác trên **đoạn bất kỳ** `[l, r]` thì `[1, 2, 1]` cần mấy thao tác?

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `[1, 4]`, `[(1, 2), (2, 3)]`, `2`, `0`, `2` (`[2,9,9,2]`: `2|9` và `9|2`). Bài 3: chỉ cần đếm số lần `nums[i] != nums[i - 1]`.
- **Mức 2 (vừa):** Bài 2: `gaps = [0, -3, 6, 0, 0, -4]`, các cặp khác nhau: `(4,1)`, `(1,7)`, `(7,3)` → đáp án `3`. Dãy thao tác phải → trái: `(p=6, k=-4)` cho `[0,0,-3,3,3,3,3]`; rồi `(p=3, k=+6)` cho `[6,6,3,3,3,3,3]`; rồi `(p=2, k=-3)` cho `[3,3,3,3,3,3,3]`. Bài 4: trong `ops_sequence`, mỗi khi `arr[i-1] != arr[i]` thì thêm `(i, arr[i] - arr[i-1])` và cập nhật tiền tố.
- **Mức 3 (sâu):** Bài 5: với hậu tố, duyệt `i` từ `0` đến `n - 2`; nếu `arr[i] != arr[i + 1]` thì cộng `arr[i] - arr[i + 1]` vào `arr[i + 1:]`. Câu hỏi thêm: `[1, 2, 1]` với đoạn bất kỳ chỉ cần `1` thao tác: chọn đoạn gồm đúng chỉ số `1`, cộng `-1` được `[1, 1, 1]`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [33]:
# Bài 1
assert [1, 4, 2][:2] == [1, 4]
assert list(zip([1, 2, 3], [1, 2, 3][1:])) == [(1, 2), (2, 3)]
assert sum([True, False, True]) == 2
assert Solution().minOperations([5, 5, 5, 5]) == 0
assert Solution().minOperations([2, 9, 9, 2]) == 2

# Bài 2
nums_b2 = [4, 4, 1, 7, 7, 7, 3]
assert gaps_of(nums_b2) == [0, -3, 6, 0, 0, -4]
assert Solution().minOperations(nums_b2) == 3
arr = list(nums_b2)
for p, k in [(6, 3 - 7), (3, 7 - 1), (2, 1 - 4)]:     # phải -> trái
    apply_prefix(arr, p, k)
assert len(set(arr)) == 1, arr

# Bài 3
def min_operations_todo_solution(nums):
    ops = 0
    for i in range(1, len(nums)):
        if nums[i] != nums[i - 1]:
            ops += 1
    return ops

assert min_operations_todo_solution([1, 4, 2]) == 2
assert min_operations_todo_solution([10, 10, 10]) == 0

# Bài 4
def min_operations_groupby2(nums):
    return len(list(groupby(nums))) - 1

def ops_sequence(nums):
    arr = list(nums)
    seq = []
    for i in range(len(arr) - 1, 0, -1):
        if arr[i - 1] != arr[i]:
            k = arr[i] - arr[i - 1]
            seq.append((i, k))
            for j in range(i):
                arr[j] += k
    return seq

rng = random.Random(19)
for _ in range(500):
    nums = [rng.randint(-5, 5) for _ in range(rng.randint(1, 15))]
    seq = ops_sequence(nums)
    assert len(seq) == min_operations_groupby2(nums) == Solution().minOperations(nums)
    arr = list(nums)
    for p, k in seq:
        apply_prefix(arr, p, k)
    assert len(set(arr)) == 1

# Bài 5
def min_ops_suffix_sim(nums):
    arr = list(nums)
    ops = 0
    for i in range(len(arr) - 1):
        if arr[i] != arr[i + 1]:
            d = arr[i] - arr[i + 1]
            for j in range(i + 1, len(arr)):
                arr[j] += d
            ops += 1
    assert len(set(arr)) == 1
    return ops

for _ in range(500):
    nums = [rng.randint(-5, 5) for _ in range(rng.randint(1, 15))]
    assert min_ops_suffix_sim(nums) == Solution().minOperations(nums)
# [1, 2, 1] với một thao tác trên đoạn [1, 1] (chỉ số 1): cộng -1 -> [1, 1, 1]
arr = [1, 2, 1]
arr[1] += -1
assert len(set(arr)) == 1
print("Tất cả đáp án đều đúng.")

Tất cả đáp án đều đúng.


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (đếm số cặp kề khác nhau)

```python
class Solution:
    def minOperations(self, nums: List[int]) -> int:
        ops = 0
        for i in range(1, len(nums)):
            if nums[i] != nums[i - 1]:
                ops += 1
        return ops
```

**Checklist**

- [ ] Hiểu: một thao tác tiền tố chỉ đổi **một** hiệu kề, nên đáp án = số cặp kề khác nhau?
- [ ] Vòng lặp bắt đầu từ `i = 1` (không đụng `nums[-1]`) và không đọc `nums[i + 1]` ở cuối?
- [ ] Không dùng `len(set(nums)) - 1` hay `n - 1`?
- [ ] Mảng 1 phần tử trả `0`, mảng toàn bằng nhau trả `0`?
- [ ] Đã thử `[1,4,2] -> 2`, `[10,10,10] -> 0`, `[1,2,1] -> 2`?
- [ ] Đã nhớ bài là Premium — đối chiếu lại đề gốc nếu có tài khoản?
- [ ] Skeleton đúng `class Solution` / `def minOperations(self, nums: List[int]) -> int`?